# TP 5 — Word2Vec : prédiction de contexte

## Rappels

Jusqu'ici, dans nos TPs, les vecteurs des mots étaient **construits** en comptant les cooccurrences (PPMI) puis en comprimant la matrice (SVD). Word2Vec change de paradigme : les vecteurs sont **appris** en entraînant un modèle à *prédire* le contexte (Mikolov et al., 2013). Le paradigme est **auto-supervisé** : le corpus lui-même sert de donnée d'entraînement, sans annotation externe.

Deux architectures :

- **CBOW** prédit le mot central à partir de son contexte : $\mathbb{P}(w_t \mid w_{t-k}, \dots, w_{t+k})$ — entraînement plus rapide, efficace pour les mots fréquents ;
- **Skip-gram** prédit chaque mot de contexte à partir du mot central : $\mathbb{P}(w_{t+j} \mid w_t)$ — plus lent, préférable pour les mots rares.

Un modèle apprend deux matrices de paramètres : $W \in \mathbb{R}^{|V| \times d}$ (vecteurs d'entrée, un par mot) et $C \in \mathbb{R}^{|V| \times d}$ (vecteurs de sortie, un par mot). Un mot possède donc **deux vecteurs** (cible / contexte) ; on garde généralement les lignes de $W$ comme représentations finales.

L'entraînement maximise la log-vraisemblance des contextes réellement observés :

$$\mathcal{L} = \sum_t \sum_{\substack{-k \leq j \leq k \\ j \neq 0}} \log \mathbb{P}(w_{t+j} \mid w_t), \qquad \mathbb{P}(c \mid w) = \frac{\exp(\mathbf{w} \cdot \mathbf{c})}{\sum_{c' \in V} \exp(\mathbf{w} \cdot \mathbf{c}')}.$$

Ce softmax coûte $|V|$ produits scalaires par paire. Le **negative sampling** le remplace par une classification binaire : pour chaque paire positive $(w, c)$ on tire $m$ mots au hasard et on minimise

$$\mathcal{L}_{NS}(w, c) = - \log \sigma(\mathbf{w} \cdot \mathbf{c}) - \sum_{i=1}^{m} \log \sigma(-\mathbf{w} \cdot \mathbf{c}_i^-), \qquad \sigma(x) = \frac{1}{1 + e^{-x}},$$

soit $1 + m$ produits scalaires au lieu de $|V|$. L'optimisation se fait par **descente de gradient** : $\theta \leftarrow \theta - \eta \nabla J(\theta)$.

Les **hyperparamètres** sont fixés avant l'entraînement : l'architecture (`sg`), la dimension `vector_size` (trop petite = sur-compression ; trop grande = sur-apprentissage sur un petit corpus), la fenêtre `window` (étroite → syntaxe, large → thème), le nombre de négatifs `negative`, le nombre d'époques `epochs`.


Pour utiliser Word2Vec, nous utiliserons la bibliothèque Python `gensim`, qu'il faut installer avec `pip` avant :

```python
from gensim.models import Word2Vec

modele = Word2Vec(sentences=phrases, vector_size=100, window=2,
                  min_count=1, sg=1, negative=5, epochs=30, seed=42)
modele.wv["chat"]                        # vecteur d'un mot, forme (100,)
modele.wv.most_similar("chat", topn=10)   # dix plus proches voisins
modele.wv.most_similar(positive=["roi", "femme"], negative=["homme"])
modele.wv.similarity("chat", "chaton")
```

Pour la partie 1, nous avons besoin de vecteurs pré-entraînés (en français) : la bibliothèque **fastText** en fournit pour de nombreuses langues, dont le français (**2 000 000 de types, 300 dimensions**, appris sur Common Crawl + Wikipédia). fastText sera étudié en détail la **semaine prochaine** ; aujourd'hui, on se contente de charger ses vecteurs et de les interroger. 

# Récupération des données et du code précédent

Exécutez ces cellules qui vous permettent de récupérer données et code des TP précédents. Vous devez obtenir **111 209 tokens**, **112 699 paires distinctes** et deux matrices de forme (2000, 2000).

Ce TP utilise en outre la bibliothèque `gensim` : si elle n'est pas installée, exécutez d'abord `%pip install gensim` dans une cellule.

In [1]:
import re
from collections import Counter
from pathlib import Path
from urllib.request import Request, urlopen
import numpy as np

url = "https://www.gutenberg.org/cache/epub/14155/pg14155.txt"
fichier = Path("corpus/madame_bovary_gutenberg_14155.txt")
if not fichier.exists():
    fichier.parent.mkdir(parents=True, exist_ok=True)
    requete = Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urlopen(requete) as reponse:
        contenu = reponse.read().decode("utf-8-sig")
    fichier.write_text(contenu, encoding="utf-8")

texte_brut = fichier.read_text(encoding="utf-8")
marque_debut = "*** START OF THE PROJECT GUTENBERG EBOOK MADAME BOVARY ***"
marque_fin = "*** END OF THE PROJECT GUTENBERG EBOOK MADAME BOVARY ***"
texte = texte_brut.split(marque_debut, 1)[1].split(marque_fin, 1)[0]

motif_francais = r"[^\W\d_]+(?:['’-][^\W\d_]+)*"
tokens = re.findall(motif_francais, texte.lower())

frequences = Counter(tokens)
vocabulaire = [mot for mot, compte in frequences.most_common(2000)]
ensemble_vocabulaire = set(vocabulaire)
mot_vers_id = {mot: i for i, mot in enumerate(vocabulaire)}
id_vers_mot = {i: mot for mot, i in mot_vers_id.items()}

print(f"{len(tokens):,} tokens, vocabulaire restreint : {len(vocabulaire)}")


111,209 tokens, vocabulaire restreint : 2000


# Partie 1 — Utiliser un modèle pré-entraîné

# Exercice 1 — Charger et inspecter les vecteurs pré-entraînés

Nous chargeons les vecteurs français distribués par fastText (`cc.fr.300`) : c'est notre « Word2Vec de production », à comparer aux petites représentations construites en TP. Le fonctionnement interne de fastText sera vu la **semaine prochaine** — pour aujourd'hui, considérez ces vecteurs comme les lignes de la matrice $W$ d'un Word2Vec entraîné sur d'immenses corpus.

Le fichier pèse environ 1,5 Go compressé (≈ 4,5 Go décompressé) : le téléchargement peut être long, lancez les deux cellules ci-dessous **le plus tôt possible** dans la séance. Si votre machine manque d'espace ou de mémoire, votre enseignante vous fournira le fichier ou une version plus légère (`wiki.fr`).

> Avec une ancienne version de `gensim` (< 4.0), le chargement se faisait via `FastText.load_fasttext_format("modele/cc.fr.300.bin")`.

In [5]:
import gzip
import shutil
from pathlib import Path
from urllib.request import Request, urlopen

url = "https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.fr.300.bin.gz"
fichier_gz = Path("modele/cc.fr.300.bin.gz")
fichier_bin = Path("modele/cc.fr.300.bin")

if not fichier_bin.exists():
    fichier_gz.parent.mkdir(parents=True, exist_ok=True)
    if not fichier_gz.exists():
        print("téléchargement de", url)
        requete = Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urlopen(requete) as reponse, fichier_gz.open("wb") as sortie:
            shutil.copyfileobj(reponse, sortie)
    with gzip.open(fichier_gz, "rb") as entree, fichier_bin.open("wb") as sortie:
        shutil.copyfileobj(entree, sortie)

print("fichier prêt :", fichier_bin)

téléchargement de https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.fr.300.bin.gz
fichier prêt : modele/cc.fr.300.bin


In [8]:
from gensim.models.fasttext import load_facebook_vectors

fasttext_fr = load_facebook_vectors("modele/cc.fr.300.bin")

print("dimension :", fasttext_fr.vector_size)
print("types :", len(fasttext_fr))

dimension : 300
types : 2000000


**Question 1.**
1. Combien de **paramètres** les matrices de représentation $W$ et $C$ représentent elles au total?
2. Récupérez les embeddings pour `roi` et `reine` (syntaxe: `fasttext_fr[mot]`).
2. Affichez leur similarité cosinus (fonction `similarite_cosinus` définie plus haut).

# Exercice 2 — Vérifier les analogies

Le CM présente la propriété qui a fait de Word2Vec une référence : certaines relations sémantiques correspondent à des **directions régulières** dans l'espace. On utilise la méthode `.most_similar(positive=..., negative=...)`.

**Question 1.**
Retournez les vecteurs les plus proches de $\overrightarrow{roi} - \overrightarrow{homme} + \overrightarrow{femme} \approx \overrightarrow{reine}$
L'analogie fonctionne-t-elle bien ?


**Question 2.**
Inventez puis testez deux autres analogies : une de type **genre** (par exemple autour de `père`/`mère`) et une de type **conjugaison** (par exemple `mange - manger + dormir` ?). Les directions sont-elles aussi régulières que pour l'analogie roi/reine ?


# Partie 2 — Entraîner son propre modèle

# Exercice 4 — Word2Vec sur Madame Bovary

On entraîne maintenant un modèle de toutes pièces sur notre corpus. 

**Rappel du CM** : un des avantages de Word2Vec est que l'entraînement est **auto-supervisé** (les « données d'exemple » sont le corpus lui-même).

**Question 1.** `Word2Vec` attend une liste de **phrases**, chaque phrase étant une liste de tokens (on va faire simple en splittant sur le mot).
1. Découpez `texte` en phrases (`re.split` sur la ponctuation forte `[.!?…]+`), puis tokenisez chaque phrase avec `motif_francais` (que vous allez chercher dans le TP2).
2. Ne conservez que les phrases d'au moins trois tokens. Combien de phrases obtenez-vous ?


**Question 2.** La cellule suivante entraîne le modèle (skip-gram, $d = 100$, fenêtre 2, cinq négatifs). Exécutez-la, puis :
1. Combien de types le vocabulaire contient-il ? Comparez aux 2 000 000 de fastText et aux 2 000 du vocabulaire restreint des TP précédents.
2. Combien de paramètres ce modèle contient-il ?

In [51]:
from gensim.models import Word2Vec

modele_bovary = Word2Vec(
    sentences=phrases_tokenisees,
    vector_size=100,   # d : dimension des vecteurs
    window=2,          # k : taille de la fenêtre de contexte
    min_count=1,       # conserver tous les types du corpus
    sg=1,              # 1 = skip-gram, 0 = CBOW
    negative=5,        # m : nombre d'exemples négatifs
    epochs=30,
    seed=42,
)

print("types :", len(modele_bovary.wv))
print("dimension :", modele_bovary.wv.vector_size)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


types : 14413
dimension : 100


**Question 3.** Refaites les mesures de la partie 1 avec `modele_bovary.wv` :
1. Affichez les dix plus proches voisins des cinq mots `["emma", "charles", "maison", "amour", "cheval"]`.
Comparez aux listes pré-entraînées sur Word2Vec "basique" (exercice 2) et PPMI. Réutilisez `jaccard` (TP 4) pour quantifier l'accord entre les trois espaces.
2. Comparez le cosinus `emma`/`charles` à celui des autres espaces : comment a-t-il évolué ?
3. Les voisins relèvent-ils surtout de la **thématique du roman** ou de la **syntaxe** ? Justifiez sur quelques exemples.

**Question 4.**
1. Testez les analogies de l'exercice 3 sur `modele_bovary` : fonctionnent-elles ?
2. Pourquoi ? Pensez à la taille du corpus d'entraînement de Mikolov et al. (des milliards de mots) devant celle de *Madame Bovary* (≈ 111 000 tokens).


# Exercice 5 — Impact des hyperparamètres

On mesure leurs effets sur les voisins de `maison`.

**Question 1.**
1. Ré-entraînez le modèle avec $d \in \{25, 50, 100, 300\}$, les autres paramètres restant inchangés.
2. Pour chaque $d$, affichez la similarité de Jaccard entre les dix voisins de `maison` dans ce modèle et ceux du modèle de référence ($d = 100$).

# Exercice 6 - Représentation graphique

Réalisez une représentation graphique des mots dans l'espace réduit (deux premières dimensions) entraîné sur $d=100$.